# 🎓 Student Performance — Data Preparation Pipeline

**Domain:** Education  
**Source:** [UCI ML Repository](https://archive.ics.uci.edu/dataset/320/student+performance)  
**Target:** `G3` — final grade (0–20, regression)  
**Students:** 1,044 (395 Math + 649 Portuguese)

Predicting student academic performance from demographic, social, and behavioral factors.

## Pipeline Steps
1. Load raw CSV data (Math & Portuguese subjects)
2. Exploratory Data Analysis
3. Data Cleaning
4. Feature Engineering
5. Prepare for ML (one-hot encode, split, scale)
6. Save processed & ML-ready files

In [ ]:
import pandas as pd
import numpy as np
import json
import pickle
import warnings
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE = Path.cwd()
RAW = BASE / 'raw'
PROCESSED = BASE / 'processed'
FEATURES = BASE / 'features'
PROCESSED.mkdir(exist_ok=True)
FEATURES.mkdir(exist_ok=True)

print('✅ Libraries loaded')

---
## 1. Load Raw Data

In [ ]:
math = pd.read_csv(RAW / 'student-mat.csv', sep=';')
math['subject'] = 'Math'

por = pd.read_csv(RAW / 'student-por.csv', sep=';')
por['subject'] = 'Portuguese'

df = pd.concat([math, por], ignore_index=True)
print(f'Math: {len(math)} rows')
print(f'Portuguese: {len(por)} rows')
print(f'Combined: {df.shape[0]} rows x {df.shape[1]} columns')

---
## 2. Exploratory Data Analysis

In [ ]:
print('=== Data Types ===')
print(df.dtypes.value_counts())
print(f'\n=== Missing Values: {df.isnull().sum().sum()} ===')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'\n=== Target Distribution (G3) ===')
print(df['G3'].describe())
print(f'\n=== Subjects ===')
print(df['subject'].value_counts().to_string())

In [ ]:
print('=== Numeric Features ===')
display(df.describe().T.style.background_gradient(cmap='Blues'))

In [ ]:
print('=== Categorical Features (value counts) ===')
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

---
## 3. Data Cleaning

In [ ]:
df_clean = df.copy()

# Standardize column names
df_clean.columns = (
    df_clean.columns.str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9_]', '_', regex=True)
    .str.replace(r'_+', '_', regex=True)
    .str.strip('_')
)

# No duplicates or missing values in this dataset
print(f'Duplicates removed: {len(df) - len(df_clean)}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
print(f'Shape after cleaning: {df_clean.shape}')

In [ ]:
# Outlier capping on absences (broad range: 0–93)
col = 'absences'
Q1, Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
n_capped = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
print(f'Capped {n_capped} outliers in absences ({lower:.0f}–{upper:.0f})')

---
## 4. Feature Engineering

In [ ]:
df_feat = df_clean.copy()

# Grade changes
df_feat['g1_to_g2_change'] = df_feat['g2'] - df_feat['g1']
df_feat['g1_to_g3_change'] = df_feat['g3'] - df_feat['g1']

# Average grade
df_feat['avg_grade'] = df_feat[['g1', 'g2', 'g3']].mean(axis=1)

# Binary flags
df_feat['has_failed'] = (df_feat['failures'] > 0).astype(int)
df_feat['high_absences'] = (df_feat['absences'] > 10).astype(int)
df_feat['low_famrel'] = (df_feat['famrel'] <= 2).astype(int)

# Effort metric
df_feat['studytime_effort'] = df_feat['studytime'] / (df_feat['g3'] + 1)

# Alcohol composite
df_feat['alcohol_score'] = df_feat['dalc'] + df_feat['walc']

new_feats = [c for c in df_feat.columns if c not in df_clean.columns]
print(f'Engineered {len(new_feats)} new features: {new_feats}')
print(f'Shape: {df_feat.shape}')

---
## 5. Prepare for ML

In [ ]:
target_col = 'g3'
df_ml = df_feat.copy()

# One-hot encode categoricals
cat_cols = df_ml.select_dtypes(include=['object']).columns.tolist()
if 'subject' in cat_cols:
    cat_cols.remove('subject')

for col in cat_cols:
    dummies = pd.get_dummies(df_ml[col], prefix=col)
    for c in dummies.columns:
        df_ml[c] = dummies[c].astype(int)
    df_ml = df_ml.drop(columns=[col])

# Features and target
numeric = df_ml.select_dtypes(include=[np.number]).columns.tolist()
numeric.remove(target_col)

X = df_ml[numeric]
y = df_ml[target_col]

print(f'Features: {len(numeric)}')
print(f'Samples: {len(X)}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_df = pd.DataFrame(X_train_scaled, columns=numeric)
X_test_df = pd.DataFrame(X_test_scaled, columns=numeric)

print(f'Train: {X_train_df.shape}')
print(f'Test:  {X_test_df.shape}')

---
## 6. Save Processed Data

In [ ]:
# Cleaned + features
clean_path = PROCESSED / 'student-performance_clean.csv'
df_feat.to_csv(clean_path, index=False)
print(f'Saved: {clean_path}')

# ML files
X_train_df.to_csv(FEATURES / 'X_train_scaled.csv', index=False)
X_test_df.to_csv(FEATURES / 'X_test_scaled.csv', index=False)
pd.DataFrame(y_train).to_csv(FEATURES / 'y_train.csv', index=False, header=[target_col])
pd.DataFrame(y_test).to_csv(FEATURES / 'y_test.csv', index=False, header=[target_col])

with open(FEATURES / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open(FEATURES / 'feature_names.json', 'w') as f:
    json.dump({'features': numeric, 'target': target_col}, f, indent=2)

print('ML files saved to features/')
print(f'  X_train_scaled.csv: {X_train_df.shape}')
print(f'  X_test_scaled.csv:  {X_test_df.shape}')
print(f'  y_train.csv:        {y_train.shape}')
print(f'  y_test.csv:         {y_test.shape}')
print(f'  scaler.pkl')
print(f'  feature_names.json')

---
## 7. Quick Model Baseline

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train_df, y_train)
preds = lr.predict(X_test_df)

print(f'Linear Regression Baseline:')
print(f'  R²:  {r2_score(y_test, preds):.4f}')
print(f'  MAE: {mean_absolute_error(y_test, preds):.2f} grade points')

---
## 8. Metadata

In [ ]:
metadata = {
    'dataset_name': 'student-performance',
    'domain': 'education',
    'created_date': str(datetime.now()),
    'source': 'UCI Machine Learning Repository — Student Performance',
    'source_url': 'https://archive.ics.uci.edu/dataset/320/student+performance',
    'license': 'CC BY 4.0',
    'description': 'Student performance in secondary education of two Portuguese schools.',
    'data_shape': {
        'total_rows': int(df_feat.shape[0]),
        'total_columns': int(df_feat.shape[1]),
        'math_students': int((df_feat[df_feat['subject'] == 'Math']).shape[0]),
        'portuguese_students': int((df_feat[df_feat['subject'] == 'Portuguese']).shape[0]),
    },
    'target_variable': 'g3',
    'ml_task': 'regression',
    'ml_data': {
        'train_samples': int(X_train_df.shape[0]),
        'test_samples': int(X_test_df.shape[0]),
        'feature_count': int(X_train_df.shape[1]),
    },
}

with open(BASE / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Saved metadata.json')
print('\n✅ Pipeline complete!')